In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm 
import warnings
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_excel('./data/que2/daily_data_new/data_1.xls', usecols=range(8))
for i in range(2, 12):
    data_read = pd.read_excel('./data/que2/daily_data_new/data_' + str(i) + '.xls', usecols=[0,1,2,3,4,5,6,7])
    data = pd.concat([data,data_read],ignore_index=True)
    del data_read
data.columns = ['stk', 'date', 'close', 'fshare', 'tshare', 'monret', 'monrf', 'pe']
data.dropna(inplace = True)
data

,stk,date,close,fshare,tshare,monret,monrf,pe
0,2,2001-01-19,14.93,509216805.0,398711877.0,0.0672,0.001650,31.27
1,2,2001-02-28,13.77,509216805.0,398711877.0,-0.0777,0.001650,28.84
2,2,2001-03-30,15.13,509216805.0,398711877.0,0.0988,0.001650,31.69
3,2,2001-04-30,14.58,509216805.0,398711877.0,-0.0364,0.001650,27.95
4,2,2001-05-31,14.20,509216805.0,398711877.0,-0.0261,0.001650,27.22
...,...,...,...,...,...,...,...,...
539492,873305,2022-03-16,25.00,0.0,0.0,0.0000,0.001971,13.48
539501,873305,2022-12-30,10.12,44819000.0,44819000.0,-0.5952,0.001931,12.32
539535,873339,2022-12-30,5.01,220203800.0,220203800.0,-0.1494,0.001931,80.18
539560,873527,2022-11-30,9.05,60042700.0,60042700.0,-0.0300,0.001676,4.98


In [3]:
data['date'] = pd.to_datetime(data['date'])
data['yearmonth'] = data['date'].dt.strftime('%Y%m').astype(int)
data['stksize'] = data['close']*data['tshare'] # 计算流通市值
data['stkep'] = 1/data['pe']
data['monexcret'] = data['monret'] - data['monrf']
data.dropna(inplace = True, subset=['stksize', 'stkep'])
# 处理数据，将后续需要的列数据存放到新dataframe中
data_pro = data[['stk','date','yearmonth','monret','stksize','stkep']]
data_pro

,stk,date,yearmonth,monret,stksize,stkep
0,2,2001-01-19,200101,0.0672,5.952768e+09,0.031980
1,2,2001-02-28,200102,-0.0777,5.490263e+09,0.034674
2,2,2001-03-30,200103,0.0988,6.032511e+09,0.031556
3,2,2001-04-30,200104,-0.0364,5.813219e+09,0.035778
4,2,2001-05-31,200105,-0.0261,5.661709e+09,0.036738
...,...,...,...,...,...,...
539492,873305,2022-03-16,202203,0.0000,0.000000e+00,0.074184
539501,873305,2022-12-30,202212,-0.5952,4.535683e+08,0.081169
539535,873339,2022-12-30,202212,-0.1494,1.103221e+09,0.012472
539560,873527,2022-11-30,202211,-0.0300,5.433864e+08,0.200803


In [4]:
# 读取beta数据
beta_data = pd.read_excel('./data/que3/beta_data/beta_1.xls', usecols=[0,1,2])
for i in range(2, 12):
    data_read = pd.read_excel('./data/que3/beta_data/beta_' + str(i) + '.xls', usecols=[0,1,2])
    beta_data = pd.concat([beta_data,data_read])
    del data_read
beta_data.columns = ['stk','date','Beta']
beta_data

,stk,date,Beta
0,2,2001-01-19,2.0023
1,2,2001-02-28,0.4679
2,2,2001-03-30,1.2196
3,2,2001-04-30,1.2584
4,2,2001-05-31,1.2285
...,...,...,...
1082,873152,2022-03-31,2.4685
1083,873152,2022-04-29,2.3646
1084,873152,2022-05-31,2.3240
1085,873152,2022-06-22,2.3230


In [5]:
# 对beta数据进行处理
beta_data['date'] = pd.to_datetime(beta_data['date'])
beta_data['yearmonth'] = beta_data['date'].dt.strftime('%Y%m').astype(int)
beta_data.dropna(inplace=True)

# 将beta数据拼接到data_pro中
data_pro_merged = pd.merge(data_pro,beta_data[['stk','yearmonth','Beta']],
                          on = ['stk','yearmonth'],
                          how = 'left'
                          )
data_pro_merged

,stk,date,yearmonth,monret,stksize,stkep,Beta
0,2,2001-01-19,200101,0.0672,5.952768e+09,0.031980,2.0023
1,2,2001-02-28,200102,-0.0777,5.490263e+09,0.034674,0.4679
2,2,2001-03-30,200103,0.0988,6.032511e+09,0.031556,1.2196
3,2,2001-04-30,200104,-0.0364,5.813219e+09,0.035778,1.2584
4,2,2001-05-31,200105,-0.0261,5.661709e+09,0.036738,1.2285
...,...,...,...,...,...,...,...
518349,873305,2022-03-16,202203,0.0000,0.000000e+00,0.074184,NaN
518350,873305,2022-12-30,202212,-0.5952,4.535683e+08,0.081169,1.9109
518351,873339,2022-12-30,202212,-0.1494,1.103221e+09,0.012472,NaN
518352,873527,2022-11-30,202211,-0.0300,5.433864e+08,0.200803,NaN


In [6]:
data_pro_merged.dropna(inplace=True)
# 对市值数据进行ln（）处理
data_pro_merged['stksize'] = np.log(data_pro_merged['stksize'])

In [7]:
month_data = np.unique(data_pro_merged['yearmonth'].values) #获取月份序列
print(len(month_data))
month_data

264


array([200101, 200102, 200103, 200104, 200105, 200106, 200107, 200108,
       200109, 200110, 200111, 200112, 200201, 200202, 200203, 200204,
       200205, 200206, 200207, 200208, 200209, 200210, 200211, 200212,
       200301, 200302, 200303, 200304, 200305, 200306, 200307, 200308,
       200309, 200310, 200311, 200312, 200401, 200402, 200403, 200404,
       200405, 200406, 200407, 200408, 200409, 200410, 200411, 200412,
       200501, 200502, 200503, 200504, 200505, 200506, 200507, 200508,
       200509, 200510, 200511, 200512, 200601, 200602, 200603, 200604,
       200605, 200606, 200607, 200608, 200609, 200610, 200611, 200612,
       200701, 200702, 200703, 200704, 200705, 200706, 200707, 200708,
       200709, 200710, 200711, 200712, 200801, 200802, 200803, 200804,
       200805, 200806, 200807, 200808, 200809, 200810, 200811, 200812,
       200901, 200902, 200903, 200904, 200905, 200906, 200907, 200908,
       200909, 200910, 200911, 200912, 201001, 201002, 201003, 201004,
      

In [8]:
# 同时放入三个自变量进行回归
import math
def T_cal(mean,std):
    T = 264
    fengzi = mean
    fengmu_pre = std/T
    fengmu = math.sqrt(fengmu_pre)
    t = fengzi / fengmu
    return t


def ols_test(month_data,data_pro_merged):
    monthly_regressions = {}
    for month in month_data:
        # 选择当前月份的数据
        monthly_data = data_pro_merged[data_pro_merged['yearmonth'] == month]
        # 对当前月份的股票数据进行回归
        X = monthly_data[['stksize', 'stkep', 'Beta']]
        y = monthly_data['monret']
        X = sm.add_constant(X)  # 添加截距项
        model = sm.OLS(y, X).fit()
        # 存储回归结果
        regression_results = pd.DataFrame(model.params, index=['const', 'stksize', 'stkep', 'Beta'])  #存储从回归模型model中提取的参数估计值
        regression_results['Month'] = month
        monthly_regressions[month] = regression_results

    # 将每个月回归获得的估计参数组成一个时间序列
    all_regressions = pd.concat([monthly_regressions[month] for month in monthly_regressions], ignore_index=True)
    
    #存储结果的列表
    const_data = []
    stksize_data = []
    stkep_data = []
    beta_data = []

    # 提取截距（const）和其他估计参数
    for i in range(0, len(all_regressions), 4):
        const_data.append(all_regressions.iloc[i][0])
        stksize_data.append(all_regressions.iloc[i + 1][0])
        stkep_data.append(all_regressions.iloc[i + 2][0])
        beta_data.append(all_regressions.iloc[i + 3][0])

    # 将提取的数据转换为DataFrame
    const_df = pd.DataFrame(const_data, columns=['const'])
    stksize_df = pd.DataFrame(stksize_data, columns=['stksize'])
    stkep_df = pd.DataFrame(stkep_data, columns=['stkep'])
    beta_df = pd.DataFrame(beta_data, columns=['Beta'])

    # 计算每个参数的平均值
    mean_const = const_df['const'].mean()
    mean_stksize = stksize_df['stksize'].mean()
    mean_stkep = stkep_df['stkep'].mean()
    mean_beta = beta_df['Beta'].mean()

    # 计算标准差
    const_std = const_df['const'].std()
    stksize_std = stksize_df['stksize'].std()
    stkep_std = stkep_df['stkep'].std()
    beta_std = beta_df['Beta'].std()

    # 计算t统计量并打印输出
    t_const = T_cal(mean_const,const_std)
    print("Mean const:", mean_const)
    print("t_const",t_const)
    
    t_stksize = T_cal(mean_stksize,stksize_std)
    print("Mean StkSize:", mean_stksize)
    print("t_stksize",t_stksize)
    
    t_stkep = T_cal(mean_stkep,stkep_std)
    print("Mean StkEP:", mean_stkep)
    print("t_stkep",t_stkep)
    
    t_beta = T_cal(mean_beta,beta_std)
    print("Mean Beta:", mean_beta)
    print("t_beta",t_beta)

In [9]:
ols_test(month_data=month_data,data_pro_merged=data_pro_merged)

Mean const: -0.16656660165390533
t_const -4.022521847736558
Mean StkSize: 0.007701023387025967
t_stksize 0.8960853137942788
Mean StkEP: -0.06919950036901706
t_stkep -2.194429754772653
Mean Beta: 0.00989935914112894
t_beta 0.5787401125224616


In [ ]:
# 如果是选择其他因子，比如两个因子组合，或者单个因子，只要OLS回归的自变量进行替换即可